# AlphaFold2 

In [9]:
import polars as pl
import os
import time

## EMBL AF accession

From accession_ids.csv from https://ftp.ebi.ac.uk/pub/databases/alphafold/README.txt:

2.) accession_ids.csv

This file contains a list of all the UniProt accessions that have predictions in
the current database version of AlphaFold DB. The file is in CSV format and
includes the following columns, separated by a comma:

* UniProt accession, e.g. A8H2R3
* First residue index (UniProt numbering), e.g. 1
* Last residue index (UniProt numbering), e.g. 199
* AlphaFold DB identifier, e.g. AF-A8H2R3-F1
* Latest version, e.g. 2

In [ ]:
remote_accid_csv = "https://ftp.ebi.ac.uk/pub/databases/alphafold/accession_ids.csv"
output_parquet = "../3_outputs/processed_data/alphafold/accession_id_mapping.parquet"
start_time = time.time()

accid_df = pl.scan_csv(
    remote_accid_csv,
    separator=',',
    has_header=False
)
accid_df = accid_df.rename({
    "column_1": "uniprot_id",
    "column_2": "first_res_idx",
    "column_3": "last_res_idx",
    "column_4": "af_db_identifier",
    "column_5": "latest_version"
})

# Add subfile index extraction (F1, F2 from AF-ID)
processed = accid_df.with_columns(
    pl.col("af_db_identifier")
    .strsplit('-').last()
    .str.extract(r"-F(\d+).*", 1)  # Capture all digits after F until end
    .alias("subfile_index")
).sort(["uniprot_id", "first_res_idx"])

print(processed.collect())

# Sort by uniprot_id and first_res_idx for interval search
processed = processed.sort(["uniprot_id", "first_res_idx"])

# Write optimized Parquet with partitioning
processed.collect().write_parquet(
    output_parquet,
    row_group_size=10000,
    compression="zstd"
)

end_time = time.time()
print(f"Processing completed in {end_time - start_time:.2f} seconds")

shape: (214_683_829, 6)
┌────────────┬───────────────┬──────────────┬──────────────────┬────────────────┬───────────────┐
│ uniprot_id ┆ first_res_idx ┆ last_res_idx ┆ af_db_identifier ┆ latest_version ┆ subfile_index │
│ ---        ┆ ---           ┆ ---          ┆ ---              ┆ ---            ┆ ---           │
│ str        ┆ i64           ┆ i64          ┆ str              ┆ i64            ┆ str           │
╞════════════╪═══════════════╪══════════════╪══════════════════╪════════════════╪═══════════════╡
│ A0A000     ┆ 1             ┆ 394          ┆ AF-A0A000-F1     ┆ 4              ┆ 1             │
│ A0A001     ┆ 1             ┆ 591          ┆ AF-A0A001-F1     ┆ 4              ┆ 1             │
│ A0A002     ┆ 1             ┆ 564          ┆ AF-A0A002-F1     ┆ 4              ┆ 1             │
│ A0A003     ┆ 1             ┆ 340          ┆ AF-A0A003-F1     ┆ 4              ┆ 1             │
│ A0A004     ┆ 1             ┆ 645          ┆ AF-A0A004-F1     ┆ 4              ┆ 1           

/tmp/ipykernel_3507313/3048715897.py:31: DeprecationWarning: The old streaming engine is being deprecated and will soon be replaced by the new streaming engine. Starting Polars version 1.23.0 and until the new streaming engine is released, the old streaming engine may become less usable. For people who rely on the old streaming engine, it is suggested to pin your version to before 1.23.0.

More information on the new streaming engine: https://github.com/pola-rs/polars/issues/20947
  processed.sink_parquet(


Processing completed in 109.52 seconds


In [ ]:
accid_df = pl.scan_parquet(output_parquet)
# accid_df = accid_df.collect()

In [28]:
accid_df["uniprot_id"].unique()

uniprot_id
str
"""A0A1E1VR23"""
"""A0A091USH9"""
"""A0A1N7AP29"""
"""A0A238K9W1"""
"""A0A3B8YFM0"""
…
"""A0A544XHK3"""
"""M3HW47"""
"""A0A1I7WQR7"""


In [45]:
# accid_df.filter(pl.col("subfile_index")!="1")
processed = accid_df.with_columns(
    pl.col("af_db_identifier")
    .str.split(by='-').list.last()
    .str.extract(r"F(\d+).*", 1)  # Capture all digits after F until end
    .alias("subfile_index")
).sort(["uniprot_id", "first_res_idx"])

In [48]:
processed.filter(~pl.col("af_db_identifier").str.ends_with("F1"))

uniprot_id,first_res_idx,last_res_idx,af_db_identifier,latest_version,subfile_index
str,i64,i64,str,i64,str


In [19]:
uniprot_ids = [f.split('-')[1] for f in os.listdir("../1_inputs/raw_inputs/alphafold/UP000005640_9606_HUMAN_v4") if f.endswith("pdb.gz")]

In [49]:
# len(set(uniprot_ids))

In [51]:
# [uid for uid in uniprot_ids if uid not in accid_df["uniprot_id"].unique()] ## Don't run, too long to run

In [30]:
accid_df.filter(pl.col("uniprot_id").is_in(uniprot_ids)).unique(subset="uniprot_id")

uniprot_id,first_res_idx,last_res_idx,af_db_identifier,latest_version,subfile_index
str,i64,i64,str,i64,str
"""P20783""",1,257,"""AF-P20783-F1""",4,"""1"""
"""Q9P0M9""",1,148,"""AF-Q9P0M9-F1""",4,"""1"""
"""Q8WW59""",1,207,"""AF-Q8WW59-F1""",4,"""1"""
"""Q8N2A8""",1,252,"""AF-Q8N2A8-F1""",4,"""1"""
"""O00562""",1,1244,"""AF-O00562-F1""",4,"""1"""
…,…,…,…,…,…
"""P17706""",1,415,"""AF-P17706-F1""",4,"""1"""
"""Q9UPQ4""",1,493,"""AF-Q9UPQ4-F1""",4,"""1"""
"""Q96MW5""",1,612,"""AF-Q96MW5-F1""",4,"""1"""
